# 01b — Extração Horária do Vento (Fortaleza)

O pipeline original agrega para mensal, perdendo a granularidade horária.
Este notebook extrai apenas os dados de Fortaleza em resolução horária
para análise de turno (dia vs noite) no projeto MOQA.

**Entrada:** CSVs brutos em `../data/raw/`  
**Saída:** `../data/processed/fortaleza_horario.parquet`

In [1]:
import pandas as pd
import glob

# Reusa a função de leitura do notebook 01
def read_station(caminho_arquivo):
    with open(caminho_arquivo, encoding='latin-1') as f:
        lines = f.readlines()

    df = pd.read_csv(caminho_arquivo, sep=';', skiprows=10, encoding='latin-1')

    df['nome_estacao'] = lines[0].split(':')[1].strip()
    df['codigo']       = lines[1].split(':')[1].strip()
    df['lat']          = float(lines[2].split(':')[1].strip())
    df['lon']          = float(lines[3].split(':')[1].strip())
    df['altitude']     = float(lines[4].split(':')[1].strip())

    return df

In [2]:
files = glob.glob('../data/raw/*.csv')
print(f"{len(files)} arquivos encontrados")

# Lê todos mas filtra só Fortaleza logo após cada leitura
# para não explodir a memória com os 16 arquivos completos
dfs = []
for f in files:
    df_temp = read_station(f)
    # Filtra Fortaleza pelo nome (verifica como aparece no seu CSV)
    if 'FORTALEZA' in df_temp['nome_estacao'].iloc[0].upper():
        dfs.append(df_temp)
        print(f"  Fortaleza encontrada: {f}")

if not dfs:
    print("ATENÇÃO: nenhum arquivo de Fortaleza encontrado.")
    print("Nomes disponíveis:", [read_station(f)['nome_estacao'].iloc[0] for f in files[:5]])

16 arquivos encontrados
  Fortaleza encontrada: ../data/raw/dados_A305_H_2010-01-01_2024-12-31.csv


In [3]:
df = pd.concat(dfs, ignore_index=True)

# Remove coluna fantasma se existir
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# Renomeia colunas
colunas = {
    'PRECIPITACAO TOTAL, HORARIO(mm)':              'precipitacao_mm',
    'RADIACAO GLOBAL(Kj/m²)':                       'radiacao_kj_m2',
    'TEMPERATURA DO AR - BULBO SECO, HORARIA(°C)':  'temperatura_bulbo_seco_c',
    'TEMPERATURA MAXIMA NA HORA ANT. (AUT)(°C)':    'temperatura_maxima_c',
    'TEMPERATURA MINIMA NA HORA ANT. (AUT)(°C)':    'temperatura_minima_c',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT)(%)':      'umidade_relativa_maxima_percent',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT)(%)':      'umidade_relativa_minima_percent',
    'UMIDADE RELATIVA DO AR, HORARIA(%)':           'umidade_relativa_percent',
    'VENTO, VELOCIDADE HORARIA(m/s)':               'vento_velocidade_m_s',
    'Data Medicao':                                 'data',
    'Hora Medicao':                                 'hora',
}
df = df.rename(columns=colunas)

# Monta datetime
df['hora'] = df['hora'].astype(str).str.zfill(4)
df['hora'] = df['hora'].str.slice(0, 2) + ':' + df['hora'].str.slice(2, 4)
df['datetime'] = pd.to_datetime(df['data'] + ' ' + df['hora'], errors='coerce')
df = df.drop(columns=['data', 'hora'])
df = df.dropna(subset=['datetime'])
df = df.sort_values('datetime').reset_index(drop=True)

print(f"Período: {df['datetime'].min()} → {df['datetime'].max()}")
print(f"Total de leituras horárias: {len(df):,}")
print(f"Nulos em vento: {df['vento_velocidade_m_s'].isna().sum()}")

Período: 2010-01-01 00:00:00 → 2024-12-31 23:00:00
Total de leituras horárias: 131,496
Nulos em vento: 18771


In [4]:
# Filtra o período do MOQA para manter os datasets alinhados
# Ajuste as datas conforme o seu moqa_data_OFICIAL.parquet
DATA_INICIO = '2023-07-01'
DATA_FIM    = '2025-11-30'

df_moqa_periodo = df[
    (df['datetime'] >= DATA_INICIO) &
    (df['datetime'] <= DATA_FIM)
].copy()

print(f"Leituras no período MOQA: {len(df_moqa_periodo):,}")
df_moqa_periodo.head()

Leituras no período MOQA: 13,200


,precipitacao_mm,RADIACAO GLOBAL(Kj/mÂ²),"TEMPERATURA DO AR - BULBO SECO, HORARIA(Â°C)",TEMPERATURA MAXIMA NA HORA ANT. (AUT)(Â°C),TEMPERATURA MINIMA NA HORA ANT. (AUT)(Â°C),umidade_relativa_maxima_percent,umidade_relativa_minima_percent,umidade_relativa_percent,vento_velocidade_m_s,nome_estacao,codigo,lat,lon,altitude,datetime
118296,0.0,-2.8,26.3,26.6,26.3,79.0,77.0,78.0,2.2,FORTALEZA,A305,-3.815833,-38.537778,29.55,2023-07-01 00:00:00
118297,0.0,-2.3,25.9,26.3,25.9,78.0,75.0,75.0,2.7,FORTALEZA,A305,-3.815833,-38.537778,29.55,2023-07-01 01:00:00
118298,0.0,-2.2,25.1,25.9,25.1,78.0,75.0,78.0,3.1,FORTALEZA,A305,-3.815833,-38.537778,29.55,2023-07-01 02:00:00
118299,0.0,-3.0,24.5,25.1,24.5,79.0,78.0,79.0,3.2,FORTALEZA,A305,-3.815833,-38.537778,29.55,2023-07-01 03:00:00
118300,0.0,-3.2,24.1,24.5,24.1,80.0,78.0,80.0,2.3,FORTALEZA,A305,-3.815833,-38.537778,29.55,2023-07-01 04:00:00


In [5]:
# Salva em parquet horário — esse é o arquivo que o notebook 03 vai consumir
df_moqa_periodo.to_parquet('../data/processed/fortaleza_horario.parquet', index=False)
print("Salvo: ../data/processed/fortaleza_horario.parquet")

Salvo: ../data/processed/fortaleza_horario.parquet
